# Deliverable 2 – Campus Coffee Shop Sales Pipeline

Linda Qviller Halvorsen

Marita Nyjordet Pedersen

In [345]:
## Imports
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

**Helper Function – Finding Data Files**

We created the find_data_file() helper function to locate input files without relying on hardcoded absolute file paths. This makes the project more flexible and allows it to work even when the project directory is located in a different place.

By keeping file path handling in a separate function, we avoid repeating code, improve readability, and make the project easier to maintain and share between group members.

In [346]:

def find_data_file(filename: str) -> Path:
    """Find a data file in shared_data/d2_data near the project."""
    current = Path.cwd().resolve()

    for parent in (current, *current.parents):
        candidate = (
            parent / "shared_data" / "d2_data" / filename
        )

        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Could not find {filename} in shared_data/d2_data "
        "under the current directory or its parents."
    )


**Task 1: File ingestion and regular expressions**

In [347]:
def read_text_file(file_path: Path) -> tuple[str, str]:
    """
    Read a text file using UTF-8 or ISO-8859-1 encoding.

    Args:
        file_path: Path to the text file.

    Returns:
        A tuple containing the file content and the encoding used.
    """
    try:
        file_text_content = file_path.read_text(encoding="utf-8")
        encoding_used = "UTF-8"

    except UnicodeDecodeError:
        file_text_content = file_path.read_text(encoding="ISO-8859-1")
        encoding_used = "ISO-8859-1"

    return file_text_content, encoding_used

In [348]:
log_pattern = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>[^\]\s]+)\] "
    r"item=(?P<item>.+?) "
    r"price=(?P<price>\d+(?:\.\d+)?) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$",
    re.MULTILINE
)

In [349]:
def parse_log_file(file_text_content: str) -> list[dict]:
    """
    Parse valid transaction lines from log text.

    Args:
        file_text_content: The complete log file content.

    Returns:
        A list of dictionaries representing valid transactions.
    """
    transactions = []

    for match in log_pattern.finditer(file_text_content):
        transaction = match.groupdict()
        transaction["price"] = float(transaction["price"])
        transactions.append(transaction)

    return transactions

**Task 2: Recursive utility functions**

In [350]:

def find_log_files(directory: str | Path) -> list[Path]:
    # Convert the input path to a Path object
    directory = Path(directory)

    # Create a list to store matching log files
    log_files = []

    # Iterate through the directory contents
    for item in directory.iterdir():

        # If the item is a directory, search recursively
        if item.is_dir():
            log_files.extend(find_log_files(item))

        # If the item is a .log file, add its Path object
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    # Return the list of Path objects
    return log_files


In [351]:

def find_active_register_ids(data: object) -> set[str]:
    """Find all active register IDs in nested dictionaries and lists."""
    register_ids: set[str] = set()

    if isinstance(data, dict):
        if data.get("active") is True and isinstance(data.get("id"), str):
            register_ids.add(data["id"])

        for value in data.values():
            register_ids.update(find_active_register_ids(value))

    elif isinstance(data, list):
        for item in data:
            register_ids.update(find_active_register_ids(item))

    return register_ids


**Limitation:** Register IDs are not necessarily unique across locations. The function identifies active register IDs without distinguishing between locations. This may cause registers from different locations to be treated as the same register, limiting location-specific analysis.

**Data Loading**

We created the load_all_transactions() helper function to separate data loading from the main workflow. The function uses read_text_file() and parse_log_file() to process each log file and combine all valid transactions into a single list.

This approach avoids code duplication, improves readability, and makes the main workflow easier to maintain. It also allows us to test the data-loading process separately before creating the Pandas DataFrame.

In [352]:

def load_all_transactions(log_files: list[Path]) -> list[dict]:
    """
    Read and parse transactions from multiple log files.

    Args:
        log_files: A list of paths to log files.

    Returns:
        A combined list of transaction dictionaries.
    """
    all_transactions = []

    for log_file in log_files:
        file_text_content, encoding_used = read_text_file(log_file)

        transactions = parse_log_file(file_text_content)

        all_transactions.extend(transactions)

    return all_transactions

**Task 3: Functions for cleaning and analysing data using NumPy**

In [353]:

def remove_price_outliers(
    prices: list[float] | np.ndarray
) -> np.ndarray:
    """
    Replace price outliers using the 3-standard-deviation rule.

    Args:
        prices: A list or NumPy array of transaction prices.

    Returns:
        A cleaned 1D NumPy float array.
    """
    # Convert input to a NumPy float array
    prices_array = np.array(prices, dtype=float)

    # Check that the array is one-dimensional
    if prices_array.ndim != 1:
        raise ValueError("Prices must be a 1D array.")

    # Handle empty input
    if prices_array.size == 0:
        return prices_array

    # Calculate mean and standard deviation
    mean_price = np.mean(prices_array)
    std_price = np.std(prices_array)

    # Identify outliers
    outliers = np.abs(prices_array - mean_price) > 3 * std_price

    # Find valid prices
    valid_prices = prices_array[~outliers]

    # Replace outliers with the median of valid prices
    if valid_prices.size > 0:
        median_price = np.median(valid_prices)
        prices_array[outliers] = median_price

    return prices_array
  

In [354]:

def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Compute the element-wise refund ratio.

    Args:
        sales_count: A NumPy array containing sales counts.
        refund_count: A NumPy array containing refund counts.

    Returns:
        A NumPy array containing the refund ratios.
    """
    refund_ratio = refund_count / (sales_count + 10**-6)

    return refund_ratio

**Task 4: Data analysis using Pandas**

In [355]:

def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Convert transaction dictionaries into a DataFrame.

    Convert timestamps to datetime, set the timestamp as
    the index, and sort transactions chronologically.

    Args:
        transactions: A list of transaction dictionaries.

    Returns:
        A DataFrame sorted by timestamp.
    """
    df = pd.DataFrame(transactions)

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    df = df.set_index("timestamp")

    df = df.sort_index()

    return df



In [356]:

def correct_nok_prices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Multiply NOK transaction prices by 10.

    Args:
        df: A DataFrame containing transaction prices and currencies.

    Returns:
        A DataFrame with corrected NOK prices.
    """
    # Create a copy to avoid modifying the original DataFrame
    corrected_df = df.copy()

    # Identify transactions in NOK
    nok_mask = corrected_df["currency"] == "NOK"

    # Correct NOK prices
    corrected_df.loc[nok_mask, "price"] *= 10.0

    return corrected_df


In [357]:

def calculate_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate hourly total price per item.

    Args:
        df: A transaction DataFrame indexed by timestamp.

    Returns:
        A DataFrame containing hourly revenue per item.
    """
    hourly_revenue = (
        df.groupby("item")
        .resample("1h")["price"]
        .sum()
        .reset_index()
    )

    return hourly_revenue


**Limitation:** This function sums prices across all transaction statuses (SUCCESS, REFUND, and CANCELLED) and currencies without conversion. Therefore, the results represent aggregated transaction values rather than actual net revenue in a single currency.

In [358]:
def create_revenue_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a pivot table of total transaction prices
    per item across status categories.

    Args:
        df: A transaction DataFrame containing
            item, price, and status columns.

    Returns:
        A pivot table with items as rows and
        status categories as columns.
    """
    pivot = pd.pivot_table(
        df,
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )

    pivot = pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"],
        fill_value=0
    )

    return pivot


**Limitation:** This pivot table sums transaction values by item and status, but does not distinguish between currencies. The values are useful for comparing transaction categories, but should not be interpreted as financial totals in a single currency.

In [359]:
def calculate_refund_ratio_by_item(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Calculate the refund ratio for each item.

    Args:
        df: A DataFrame containing item and status columns.

    Returns:
        A DataFrame containing sales count, refund count,
        and refund ratio for each item.
    """
    items = sorted(df["item"].unique())

    sales_count = (
        df[df["status"] == "SUCCESS"]
        .groupby("item")
        .size()
        .reindex(items, fill_value=0)
    )

    refund_count = (
        df[df["status"] == "REFUND"]
        .groupby("item")
        .size()
        .reindex(items, fill_value=0)
    )

    refund_report = pd.DataFrame({
        "sales_count": sales_count,
        "refund_count": refund_count
    })

    refund_report["refund_ratio"] = compute_refund_ratio(
        refund_report["sales_count"].to_numpy(),
        refund_report["refund_count"].to_numpy()
    )

    refund_report.index.name = "item"

    return refund_report

**Task 5: Create full workflow**

In [360]:

def run_workflow() -> None:
    """
    Run the complete Campus Coffee Shop sales pipeline.

    Discover and parse log files, clean prices, correct NOK
    values, generate reports, and save the required outputs.
    """

    # Step 1: Find log files and active register IDs
    data_directory = find_data_file("store_config.json").parent
    log_files = find_log_files(data_directory)

    with open(data_directory / "store_config.json",
              "r", encoding="utf-8") as config_file:
        store_config = json.load(config_file)

    active_register_ids = find_active_register_ids(store_config)

    print("Log files found:", len(log_files))
    print("Active register IDs:", sorted(active_register_ids))

    # Step 2: Read and parse all log files
    all_transactions = load_all_transactions(log_files)

    print("Transactions parsed:", len(all_transactions))

    if not all_transactions:
        raise ValueError("No valid transactions were found.")

    # Step 3: Create the initial DataFrame
    df = create_transaction_dataframe(all_transactions)

    # Step 4: Clean price outliers using NumPy
    df["price"] = remove_price_outliers(
        df["price"].to_numpy(copy=True)
    )

    # Step 5: Correct NOK prices
    df = correct_nok_prices(df)

    # Step 6: Generate reports
    hourly_revenue = calculate_hourly_revenue(df)
    revenue_pivot = create_revenue_pivot(df)
    refund_report = calculate_refund_ratio_by_item(df)

    # Step 7: Create the output directory
    output_directory = Path.cwd() / "output"
    output_directory.mkdir(parents=True, exist_ok=True)

    # Save cleaned transactions
    df.to_csv(
        output_directory / "cleaned_transactions.csv"
    )

    # Save hourly revenue as JSON
    hourly_revenue.to_json(
        output_directory / "hourly_revenue.json",
        orient="records",
        date_format="iso",
        indent=4
    )

    # Save status pivot table
    revenue_pivot.to_csv(
        output_directory / "status_pivot_table.csv"
    )

    # Save refund ratios
    refund_report.to_csv(
        output_directory / "item_refund_ratios.csv"
    )

    print("Workflow completed successfully.")
    print("Output directory:", output_directory)


**Run Full Workflow**

In [361]:
run_workflow()

Log files found: 9
Active register IDs: ['POS_01', 'POS_02', 'POS_03']
Transactions parsed: 720
Workflow completed successfully.
Output directory: c:\Users\Bruker\INF_prosjekter\Deliverable_02_week_43\output


**Testing and Validation**

The following tests verify the individual functions and their integration.
The tests use separate variables to avoid modifying the original transaction data.
We use assert statements to verify that the functions produce expected results.
If all assertions pass, a PASS message is displayed. If a test fails, Python raises an AssertionError.
Passing tests confirms correct behavior for the tested cases, but does not guarantee error-free code.

In [362]:

# Test file reading and log parsing
test_log_path = find_data_file(
    "2026/library_cafe/POS_01/register_sales.log"
)

test_content, test_encoding = read_text_file(test_log_path)
test_transactions = parse_log_file(test_content)

assert isinstance(test_content, str)
assert test_encoding in ("UTF-8", "ISO-8859-1")
assert isinstance(test_transactions, list)
assert len(test_transactions) > 0

expected_fields = {
    "timestamp", "register_id", "item",
    "price", "currency", "status"
}

assert all(
    expected_fields.issubset(transaction.keys())
    for transaction in test_transactions
)

print("Encoding:", test_encoding)
print("Valid transactions:", len(test_transactions))

display(pd.DataFrame(test_transactions).head())

print("PASS: File reading and log parsing")



Encoding: UTF-8
Valid transactions: 80


,timestamp,register_id,item,price,currency,status
0,2026-10-15 07:00:38,POS_01,Croissant,3.98,EUR,SUCCESS
1,2026-10-15 07:02:55,POS_01,Sandwich,7.70,USD,SUCCESS
2,2026-10-15 07:05:45,POS_01,Sandwich,7.43,USD,SUCCESS
3,2026-10-15 07:07:17,POS_01,Cappuccino,3.72,USD,SUCCESS
4,2026-10-15 07:08:34,POS_01,Muffin,3.45,EUR,SUCCESS


PASS: File reading and log parsing


In [363]:

# Test recursive file search and active register IDs
test_data_directory = find_data_file("store_config.json").parent

test_log_files = find_log_files(test_data_directory)

with open(
    test_data_directory / "store_config.json",
    "r",
    encoding="utf-8"
) as test_config_file:
    test_config = json.load(test_config_file)

test_active_ids = find_active_register_ids(test_config)

assert isinstance(test_log_files, list)
assert all(isinstance(path, Path) for path in test_log_files)
assert len(test_log_files) > 0
assert isinstance(test_active_ids, set)

print("Log files found:", len(test_log_files))
print("Active register IDs:", sorted(test_active_ids))

display(pd.DataFrame({
    "Log file": [str(path) for path in test_log_files]
}))

print("PASS: Recursive functions")



Log files found: 9
Active register IDs: ['POS_01', 'POS_02', 'POS_03']


,Log file
0,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
1,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
2,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
3,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
4,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
5,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
6,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
7,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...
8,C:\Users\Bruker\INF_prosjekter\shared_data\d2_...


PASS: Recursive functions


In [364]:


# Test loading transactions from multiple log files
test_all_transactions = load_all_transactions(test_log_files)

assert isinstance(test_all_transactions, list)
assert len(test_all_transactions) > 0
assert all(
    isinstance(transaction, dict)
    for transaction in test_all_transactions
)

test_transactions_df = pd.DataFrame(test_all_transactions)

print("Total transactions:", len(test_all_transactions))

print("\nTransactions by status:")
display(
    test_transactions_df["status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Count")
)

print("\nSample transactions:")
display(test_transactions_df.head())

print("PASS: Data loading")



Total transactions: 720

Transactions by status:


,Status,Count
0,SUCCESS,467
1,CANCELLED,128
2,REFUND,125



Sample transactions:


,timestamp,register_id,item,price,currency,status
0,2026-10-15 07:00:38,POS_01,Croissant,3.98,EUR,SUCCESS
1,2026-10-15 07:02:55,POS_01,Sandwich,7.70,USD,SUCCESS
2,2026-10-15 07:05:45,POS_01,Sandwich,7.43,USD,SUCCESS
3,2026-10-15 07:07:17,POS_01,Cappuccino,3.72,USD,SUCCESS
4,2026-10-15 07:08:34,POS_01,Muffin,3.45,EUR,SUCCESS


PASS: Data loading


In [365]:

# Test price outlier removal
test_prices = np.array([4.50] * 20 + [999.00])

test_cleaned_prices = remove_price_outliers(test_prices)

assert isinstance(test_cleaned_prices, np.ndarray)
assert test_cleaned_prices.ndim == 1
assert test_cleaned_prices[-1] == 4.50
assert test_prices[-1] == 999.00

print("Price cleaning comparison:")

display(pd.DataFrame({
    "Original price": test_prices,
    "Cleaned price": test_cleaned_prices
}).tail())

# Test refund ratio calculation
test_sales = np.array([100, 50, 20])
test_refunds = np.array([5, 2, 4])

test_ratios = compute_refund_ratio(
    test_sales, test_refunds
)

assert np.allclose(test_ratios, [0.05, 0.04, 0.20])

print("\nRefund ratio comparison:")

display(pd.DataFrame({
    "Sales": test_sales,
    "Refunds": test_refunds,
    "Refund ratio": test_ratios
}).round(4))

print("PASS: NumPy functions")



Price cleaning comparison:


,Original price,Cleaned price
16,4.5,4.5
17,4.5,4.5
18,4.5,4.5
19,4.5,4.5
20,999.0,4.5



Refund ratio comparison:


,Sales,Refunds,Refund ratio
0,100,5,0.05
1,50,2,0.04
2,20,4,0.20


PASS: NumPy functions


In [366]:

# Test DataFrame creation
test_df = create_transaction_dataframe(
    test_all_transactions
)

assert isinstance(test_df, pd.DataFrame)
assert isinstance(test_df.index, pd.DatetimeIndex)
assert test_df.index.is_monotonic_increasing

print("Transaction DataFrame:")
display(test_df.head())

# Test NOK price correction
test_currency_df = pd.DataFrame({
    "item": ["Coffee", "Tea", "Latte"],
    "price": [10.0, 5.0, 8.0],
    "currency": ["NOK", "EUR", "NOK"]
})

test_corrected_df = correct_nok_prices(test_currency_df)

nok_mask = test_currency_df["currency"] == "NOK"

assert np.allclose(
    test_corrected_df.loc[nok_mask, "price"],
    test_currency_df.loc[nok_mask, "price"] * 10
)

assert np.allclose(
    test_corrected_df.loc[~nok_mask, "price"],
    test_currency_df.loc[~nok_mask, "price"]
)

assert test_currency_df["price"].tolist() == [10.0, 5.0, 8.0]

print("\nCurrency correction comparison:")

display(pd.DataFrame({
    "Item": test_currency_df["item"],
    "Currency": test_currency_df["currency"],
    "Original price": test_currency_df["price"],
    "Corrected price": test_corrected_df["price"]
}))

print("PASS: Pandas and currency correction")



Transaction DataFrame:


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:38,POS_01,Croissant,3.98,EUR,SUCCESS
2026-10-15 07:01:09,POS_02,Cappuccino,3.82,EUR,CANCELLED
2026-10-15 07:01:17,POS_01,Muffin,3.06,EUR,SUCCESS
2026-10-15 07:01:23,POS_02,Muffin,3.84,NOK,SUCCESS
2026-10-15 07:01:56,POS_02,Sandwich,7.17,USD,SUCCESS



Currency correction comparison:


,Item,Currency,Original price,Corrected price
0,Coffee,NOK,10.0,100.0
1,Tea,EUR,5.0,5.0
2,Latte,NOK,8.0,80.0


PASS: Pandas and currency correction


In [367]:

# Prepare data for report testing
test_report_df = test_df.copy()

test_report_df["price"] = remove_price_outliers(
    test_report_df["price"].to_numpy(copy=True)
)

test_report_df = correct_nok_prices(test_report_df)

# Test hourly revenue
test_hourly_revenue = calculate_hourly_revenue(
    test_report_df
)

assert isinstance(test_hourly_revenue, pd.DataFrame)
assert {"item", "timestamp", "price"}.issubset(
    test_hourly_revenue.columns
)

print("Hourly revenue:")
display(test_hourly_revenue.head(10))

# Test revenue pivot table
test_revenue_pivot = create_revenue_pivot(
    test_report_df
)

assert isinstance(test_revenue_pivot, pd.DataFrame)
assert list(test_revenue_pivot.columns) == [
    "SUCCESS", "REFUND", "CANCELLED"
]

print("\nRevenue pivot table:")
display(test_revenue_pivot)

# Test refund ratios by item
test_refund_report = calculate_refund_ratio_by_item(
    test_report_df
)

assert isinstance(test_refund_report, pd.DataFrame)
assert {
    "sales_count", "refund_count", "refund_ratio"
}.issubset(test_refund_report.columns)

assert test_refund_report["refund_ratio"].ge(0).all()

print("\nRefund ratios by item:")
display(test_refund_report.round(4))

print("PASS: Report generation")


Hourly revenue:


,item,timestamp,price
0,Cappuccino,2026-10-15 07:00:00,849.89
1,Cappuccino,2026-10-15 08:00:00,860.27
2,Cappuccino,2026-10-15 09:00:00,192.72
3,Croissant,2026-10-15 07:00:00,736.03
4,Croissant,2026-10-15 08:00:00,769.71
5,Croissant,2026-10-15 09:00:00,98.86
6,Espresso,2026-10-15 07:00:00,619.41
7,Espresso,2026-10-15 08:00:00,523.27
8,Espresso,2026-10-15 09:00:00,109.30
9,Latte,2026-10-15 07:00:00,983.06



Revenue pivot table:


status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,1262.02,196.23,444.63
Croissant,1062.22,422.21,120.17
Espresso,803.08,197.67,251.23
Latte,1174.21,455.85,291.60
Muffin,1356.43,235.23,226.61
Sandwich,1799.02,547.89,650.23



Refund ratios by item:


,sales_count,refund_count,refund_ratio
item,,,
Cappuccino,82,18,0.2195
Croissant,85,25,0.2941
Espresso,65,20,0.3077
Latte,73,19,0.2603
Muffin,90,20,0.2222
Sandwich,72,23,0.3194


PASS: Report generation
